# 01 · Raw → Bronze
Reads the tab-separated files from the ADLS `raw` container and writes them **as received** (all columns kept as strings) to Delta tables in the `bronze` schema. Only column names are made Delta-safe, and two metadata columns are added: `ingestion_timestamp` and `source_file_name`.

Every table is reconciled against its source file row count.

In [0]:
%run ../../config/pipeline_config

In [0]:
%run ../../utils/common_functions

In [0]:
load_summary = []

for dataset in SELECTED_DATASETS:
    cfg = DATASETS[dataset]
    source = f"{RAW_PATH}{cfg['source_file']}"

    raw_df = spark.read.options(**RAW_CSV_OPTIONS).csv(source)   # no schema inference: bronze keeps raw values
    bronze_df = sanitize_column_names(add_ingestion_metadata(raw_df))

    bronze_rows = write_delta_table(bronze_df, f"{BRONZE_SCHEMA}.{dataset}", f"{BRONZE_PATH}{cfg['folder']}")

    source_rows = spark.read.options(**RAW_CSV_OPTIONS).csv(source).count()
    if bronze_rows != source_rows:
        raise ValueError(f"{dataset}: bronze has {bronze_rows} rows but source file has {source_rows}")
    load_summary.append((dataset, cfg["source_file"], source_rows, bronze_rows))

display(spark.createDataFrame(load_summary, "dataset STRING, source_file STRING, source_rows LONG, bronze_rows LONG"))